In [1]:
# Cargar el paquete LPE
library(LPE)


# Preparacion de los datos

In [6]:
# 2. Cargar datos ya normalizados (log2 + quantile norm)
data_lpe <- read.csv("datacruda_V0.1_log2_quantilenorm.csv", stringsAsFactors = FALSE)

In [8]:
# 4. Extraer matrices de expresión
expr_WT <- as.matrix(data_lpe[, grupo1_cols])
expr_TLR4 <- as.matrix(data_lpe[, grupo2_cols])

In [9]:
# 5. Calcular baseline variance para cada grupo (baseOlig.error)
# q=0.01 crea 100 bins/quantiles (puedes ajustar si querés)
bv_WT <- baseOlig.error(expr_WT, q = 0.01)
bv_TLR4 <- baseOlig.error(expr_TLR4, q = 0.01)

#### Expresión diferencial

In [10]:
# 6. Ejecutar el LPE test pasando las varianzas estimadas y nombres de probes
# Usamos los nombres de las filas (genes) como probe.set.name
lpe_result <- lpe(
  x = expr_WT,
  y = expr_TLR4,
  basevar.x = bv_WT,
  basevar.y = bv_TLR4,
  probe.set.name = data_lpe$ProbeID
)

In [12]:
# Este objeto ya es un data.frame con p-values, fold changes, etc.
head(lpe_result)
# 7. Añadir resultados al dataframe original para guardar
data_lpe$LPE_statistic <- lpe_result$statistic
data_lpe$LPE_pvalue <- lpe_result$p.value



,x.WT_1.gProcessedSignal,x.WT_2.gProcessedSignal,median.1,std.dev.1,p.outlier.x,flag.outlier.x,y.RNA5_TLR4_1.gProcessedSignal,y.RNA5_TLR4_2.gProcessedSignal,median.2,std.dev.2,p.outlier.y,flag.outlier.y,median.diff,pooled.std.dev,z.stats
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>
A_19_P00315452,10.589535,11.033535,10.811535,0.1652226,0.056926922,.,9.990152,9.906309,9.948231,0.11285672,0.59728545,.,0.8633041,0.1414835,6.101802
A_19_P00315493,4.588821,5.960011,5.274416,0.3105251,0.001775648,**,6.599604,6.413724,6.506664,0.18281690,0.47317463,.,-1.2322479,0.2548018,-4.836104
A_19_P00315502,7.166882,7.771754,7.469318,0.2500386,0.084831362,.,5.996803,5.508821,5.752812,0.21400592,0.10858791,.,1.7165059,0.2327207,7.375820
A_19_P00315543,6.080101,5.870934,5.975517,0.2840457,0.604740512,.,5.576464,5.237739,5.407102,0.23133617,0.30201642,.,0.5684157,0.2590351,2.194358
A_19_P00315551,5.856952,5.452845,5.654899,0.2950787,0.334889754,.,6.427738,6.343293,6.385515,0.18720429,0.75045674,.,-0.7306167,0.2471001,-2.956765
A_19_P00315581,12.512100,12.387483,12.449792,0.1398435,0.527808253,.,12.546988,12.761979,12.654483,0.08506404,0.07257731,.,-0.2046917,0.1157413,-1.768528


In [13]:
nrow(lpe_result)

[1] 24298

In [17]:
str(lpe_result)

'data.frame':	24298 obs. of  15 variables:
 $ x.WT_1.gProcessedSignal       : num  10.59 4.59 7.17 6.08 5.86 ...
 $ x.WT_2.gProcessedSignal       : num  11.03 5.96 7.77 5.87 5.45 ...
 $ median.1                      : num  10.81 5.27 7.47 5.98 5.65 ...
 $ std.dev.1                     : num  0.165 0.311 0.25 0.284 0.295 ...
 $ p.outlier.x                   : num  0.05693 0.00178 0.08483 0.60474 0.33489 ...
 $ flag.outlier.x                : chr  "." "**" "." "." ...
 $ y.RNA5_TLR4_1.gProcessedSignal: num  9.99 6.6 6 5.58 6.43 ...
 $ y.RNA5_TLR4_2.gProcessedSignal: num  9.91 6.41 5.51 5.24 6.34 ...
 $ median.2                      : num  9.95 6.51 5.75 5.41 6.39 ...
 $ std.dev.2                     : num  0.113 0.183 0.214 0.231 0.187 ...
 $ p.outlier.y                   : num  0.597 0.473 0.109 0.302 0.75 ...
 $ flag.outlier.y                : chr  "." "." "." "." ...
 $ median.diff                   : num  0.863 -1.232 1.717 0.568 -0.731 ...
 $ pooled.std.dev                : num  0.1

In [20]:
# Extraemos todas las columnas de lpe_result en un data.frame
lpe_df <- as.data.frame(lpe_result, stringsAsFactors = FALSE)

# Armamos el data.frame final con ProbeID, Gene_Symbol y luego todas las columnas de lpe_result
resultados_finales <- data.frame(
  ProbeID = data_lpe$ProbeID,
  Gene_Symbol = data_lpe$Gene_Symbol,
  lpe_df,
  stringsAsFactors = FALSE
)

In [21]:
write.csv(resultados_finales, "LPE_results_TLR4vsWT.csv", row.names = FALSE)

# Calcular p-valor a partir del estadistico Z

In [22]:

# 1. Calcular p-valor bilateral desde z.stats
p_values <- 2 * pnorm(-abs(lpe_result$z.stats))

In [23]:
# 2. Ajustar p-valores por múltiples pruebas (FDR)
p_adj <- p.adjust(p_values, method = "BH")

In [24]:
# 3. Calcular fold change lineal con la regla:
#    si median.diff < 0, fc = -1 / (2^abs(median.diff))
#    si median.diff >= 0, fc = 2^(median.diff)
fc_linear <- ifelse(
  lpe_result$median.diff < 0,
  -1 / (2^abs(lpe_result$median.diff)),
  2^(lpe_result$median.diff)
)

In [25]:
# 4. Armar data frame con resultados
resultados_finales <- data.frame(
  ProbeID = data_lpe$ProbeID,
  Gene_Symbol = data_lpe$Gene_Symbol,
  median_diff_log2fc = lpe_result$median.diff,
  fold_change = fc_linear,
  z_stat = lpe_result$z.stats,
  p_value = p_values,
  p_value_adj = p_adj,
  stringsAsFactors = FALSE
)

# 5. Guardar resultados en CSV
write.csv(resultados_finales, "resultados_con_pval.csv", row.names = FALSE)

In [29]:
# Asegurarse de que los datos están cargados
# Si ya tenés `resultados_finales` creado, filtramos directo

significativos <- subset(
  resultados_finales,
  abs(fold_change) >= 1.5 & p_value < 0.05
)

# Ver cuántos genes pasan el filtro
cat("Genes significativos:", nrow(significativos), "\n")

Genes significativos: 1519 


# Pruebo con parámetros más sensibles

In [31]:
bv_WT_v2 <- baseOlig.error(expr_WT, q = 0.005)
bv_TLR4_v2 <- baseOlig.error(expr_TLR4, q = 0.005)

# 2. Ejecutar el LPE test nuevamente con esas varianzas
lpe_result_v2 <- lpe(
  x = expr_WT,
  y = expr_TLR4,
  basevar.x = bv_WT_v2,
  basevar.y = bv_TLR4_v2,
  probe.set.name = rownames(data_lpe)
)

# 3. Calcular p-valor bilateral desde z.stats
p_values_v2 <- 2 * pnorm(-abs(lpe_result_v2$z.stats))

# 4. Ajustar p-valores por FDR
p_adj_v2 <- p.adjust(p_values_v2, method = "BH")

# 5. Calcular fold change lineal desde median.diff
fc_linear_v2 <- ifelse(
  lpe_result_v2$median.diff < 0,
  -1 / (2^abs(lpe_result_v2$median.diff)),
  2^(lpe_result_v2$median.diff)
)

# 6. Unir todos los resultados en un nuevo data frame
resultados_finales_v2 <- data.frame(
  ProbeID = data_lpe$ProbeID,
  Gene_Symbol = data_lpe$Gene_Symbol,
  median_diff_log2fc = lpe_result_v2$median.diff,
  fold_change = fc_linear_v2,
  z_stat = lpe_result_v2$z.stats,
  p_value = p_values_v2,
  p_value_adj = p_adj_v2,
  stringsAsFactors = FALSE
)

# 7. Aplicar el filtro Macrogen-style: |fc| ≥ 1.5 y p < 0.05
significativos_v2 <- subset(
  resultados_finales_v2,
  abs(fold_change) >= 1.5 & p_value < 0.05
)


In [32]:
# 9. Mostrar cuántos genes pasaron el filtro
cat("Genes significativos con q=0.005 y df=15:", nrow(significativos_v2), "\n")

Genes significativos con q=0.005 y df=15: 1519 
